# 3a. DTPM + SVM (Backward Compatible)

This notebook keeps the original DTPM + SVM pipeline, separated from DCNN training.
Use this when you already have trained DCNN checkpoints and want the legacy section-3 flow.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from utility import EMOTION_ENG_MAP

In [ ]:
CURR_DIR = os.getcwd()

# Update these two values before running.
DATASET = "processed_emodb_speaker_norm_loso_aug"
SPLIT_MODE = "loso"  # "original" or "loso"

DATASET_PATH = os.path.join(CURR_DIR, DATASET)
MODEL_DIR = os.path.join(CURR_DIR, "models")
RESULTS_DIR = os.path.join(CURR_DIR, "results", DATASET)
os.makedirs(RESULTS_DIR, exist_ok=True)

DCNN_PATH = os.path.join(MODEL_DIR, f"dcnn_{DATASET}.pth")
SVM_PATH = os.path.join(MODEL_DIR, f"svm_{DATASET}.joblib")

if SPLIT_MODE == "loso":
    FOLD_NAMES = sorted([
        n for n in os.listdir(DATASET_PATH)
        if n.startswith("fold") and os.path.isdir(os.path.join(DATASET_PATH, n))
    ])
else:
    FOLD_NAMES = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((227, 227), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"DATASET_PATH: {DATASET_PATH}")
print(f"SPLIT_MODE: {SPLIT_MODE}")
if SPLIT_MODE == "loso":
    print(f"Folds: {FOLD_NAMES}")

In [ ]:
def build_fc7_extractor(model_path):
    feature_extractor = models.alexnet()
    num_ftrs = feature_extractor.classifier[6].in_features
    feature_extractor.classifier[6] = nn.Linear(num_ftrs, 7)
    feature_extractor.load_state_dict(torch.load(model_path, map_location=device))
    feature_extractor.classifier = nn.Sequential(*list(feature_extractor.classifier.children())[:5])
    feature_extractor.eval()
    return feature_extractor.to(device)

def extract_fc7(segments_array, feature_extractor):
    processed = []
    for img in segments_array:
        img_copy = img.copy()
        for c in range(3):
            mn, mx = img_copy[c].min(), img_copy[c].max()
            img_copy[c] = (img_copy[c] - mn) / (mx - mn) if mx > mn else 0.0
        img_transposed = img_copy.transpose(1, 2, 0)
        processed.append(transform(img_transposed).float().unsqueeze(0))

    batch = torch.cat(processed).to(device=device, dtype=torch.float32)
    with torch.no_grad():
        return feature_extractor(batch).cpu().numpy()

def lp_norm_pooling(matrix, p=1.12):
    return np.power(np.mean(np.power(np.abs(matrix), p), axis=0), 1/p)

def dtpm(segment_features, L=2):
    N = segment_features.shape[0]
    final_feature = []
    final_feature.append(lp_norm_pooling(segment_features) * (1 / (2**L)))

    if N >= 2:
        mid = N // 2
        final_feature.append(lp_norm_pooling(segment_features[:mid]) * (1 / (2**L)))
        final_feature.append(lp_norm_pooling(segment_features[mid:]) * (1 / (2**L)))
    else:
        final_feature.extend([np.zeros(4096)] * 2)

    if L == 2:
        if N >= 4:
            q_size = N // 4
            for i in range(4):
                start = i * q_size
                end = (i + 1) * q_size if i != 3 else N
                final_feature.append(lp_norm_pooling(segment_features[start:end]) * (1 / (2**(L-1))))
        else:
            final_feature.extend([np.zeros(4096)] * 4)

    return np.concatenate(final_feature)

def process_dataset_to_global_features(dataset_dir, split_name, feature_extractor):
    X_segs = np.load(os.path.join(dataset_dir, f"X_{split_name}.npy"))
    y_lbls = np.load(os.path.join(dataset_dir, f"y_{split_name}.npy"))
    u_ids = np.load(os.path.join(dataset_dir, f"utterance_ids_{split_name}.npy"))

    utterance_dict = {}
    for i, uid in enumerate(u_ids):
        if uid not in utterance_dict:
            utterance_dict[uid] = {'segments': [], 'label': y_lbls[i]}
        utterance_dict[uid]['segments'].append(X_segs[i])

    X_global, y_global = [], []
    for uid, data in utterance_dict.items():
        fc7_feats = extract_fc7(np.array(data['segments']), feature_extractor)
        global_feat = dtpm(fc7_feats, L=2)
        X_global.append(global_feat)
        y_global.append(data['label'])

    return np.array(X_global), np.array(y_global)

In [ ]:
svm_logs = []
def log_svm(msg):
    print(msg)
    svm_logs.append(str(msg))

svm_all_true = []
svm_all_pred = []

if SPLIT_MODE == "original":
    fc7_extractor = build_fc7_extractor(DCNN_PATH)
    X_train_global, y_train_global = process_dataset_to_global_features(DATASET_PATH, "train", fc7_extractor)
    X_test_global, y_test_global = process_dataset_to_global_features(DATASET_PATH, "test", fc7_extractor)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_global)
    X_test_scaled = scaler.transform(X_test_global)

    svm_classifier = SVC(kernel='linear', decision_function_shape='ovo')
    svm_classifier.fit(X_train_scaled, y_train_global)
    predictions = svm_classifier.predict(X_test_scaled)

    svm_all_true = y_test_global.tolist()
    svm_all_pred = predictions.tolist()

    joblib.dump(svm_classifier, SVM_PATH)
    joblib.dump(scaler, os.path.join(MODEL_DIR, f"scaler_{DATASET}.joblib"))

else:
    for fold_name in FOLD_NAMES:
        fold_path = os.path.join(DATASET_PATH, fold_name)
        fold_dcnn_path = os.path.join(MODEL_DIR, f"dcnn_{DATASET}_{fold_name}.pth")
        if not os.path.exists(fold_dcnn_path):
            raise FileNotFoundError(f"Missing checkpoint: {fold_dcnn_path}")

        log_svm('-' * 70)
        log_svm(f"Fold: {fold_name}")

        fold_extractor = build_fc7_extractor(fold_dcnn_path)
        X_train_global, y_train_global = process_dataset_to_global_features(fold_path, "train", fold_extractor)
        X_test_global, y_test_global = process_dataset_to_global_features(fold_path, "test", fold_extractor)

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_global)
        X_test_scaled = scaler.transform(X_test_global)

        fold_svm = SVC(kernel='linear', decision_function_shape='ovo')
        fold_svm.fit(X_train_scaled, y_train_global)
        fold_predictions = fold_svm.predict(X_test_scaled)

        fold_acc = accuracy_score(y_test_global, fold_predictions)
        log_svm(f"Fold utterance-level SVM accuracy: {fold_acc * 100:.2f}%")

        svm_all_true.extend(y_test_global.tolist())
        svm_all_pred.extend(fold_predictions.tolist())

        joblib.dump(fold_svm, os.path.join(MODEL_DIR, f"svm_{DATASET}_{fold_name}.joblib"))
        joblib.dump(scaler, os.path.join(MODEL_DIR, f"scaler_{DATASET}_{fold_name}.joblib"))

svm_y_true_final = np.array(svm_all_true)
svm_y_pred_final = np.array(svm_all_pred)
svm_acc_final = accuracy_score(svm_y_true_final, svm_y_pred_final)
svm_report_final = classification_report(svm_y_true_final, svm_y_pred_final)

log_svm('=' * 70)
log_svm(f"Final SVM utterance-level accuracy ({SPLIT_MODE}): {svm_acc_final * 100:.2f}%")
print(svm_report_final)
svm_logs.append(svm_report_final)

svm_report_path = os.path.join(RESULTS_DIR, f"classification_report_{DATASET}.txt")
with open(svm_report_path, "w") as f:
    for line in svm_logs:
        f.write(f"{line}\n")
print(f"Saved classification log to: {svm_report_path}")

In [ ]:
EMOTION_NAMES = [EMOTION_ENG_MAP[i] for i in range(len(EMOTION_ENG_MAP))]
cm = confusion_matrix(svm_y_true_final, svm_y_pred_final, labels=np.arange(len(EMOTION_NAMES)))

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTION_NAMES, yticklabels=EMOTION_NAMES)
plt.title(f'SVM Confusion Matrix ({SPLIT_MODE.upper()})', fontsize=14, pad=15)
plt.xlabel('Predicted Emotion', fontsize=12)
plt.ylabel('True Emotion', fontsize=12)
plt.tight_layout()

svm_cm_path = os.path.join(RESULTS_DIR, f"confusion_svm_{DATASET}.png")
plt.savefig(svm_cm_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved SVM confusion matrix to: {svm_cm_path}")